In [0]:
%run "/Workspace/Repos/Git Repo/dwh_sql_repo/Medallion Architecture/Generic Framework and 3layer data load/Generic function"

In [0]:
SILVER = "/Volumes/datalake_prod/datalake_prod_schema/silver"
GOLD = "/Volumes/datalake_prod/datalake_prod_schema/gold"
GOLDDB = "datalake_prod.datalake_prod_schema"
staff = spark.read.format("delta").load(f"{SILVER}/staff")
shipments = spark.read.format("delta").load(f"{SILVER}/shipments")

joined = join_df(staff,shipments, "inner", "shipment_id")

gold_core = joined.select(
    "shipment_id",
    mask_name("staff_full_name").alias("masked_staff_name"),
    "role",
    "origin_hub_city",
    "shipment_cost",
    "shipment_year",
    "shipment_month",
    "route_segment",
    "cost_per_kg",
    "tax_amount",
    "ingestion_timestamp"
)

write_file(gold_core,f"{GOLD}/core_curated")
write_table(gold_core,f"{GOLDDB}.core_curated_tbl", mode="overwrite")

In [0]:
from pyspark.sql.window import Window
df = spark.read.format("delta").load(f"{GOLD}/core_curated")
df.show()
# TOP 3 DRIVERS BY COST PER HUB
w = Window.partitionBy("origin_hub_city") \
          .orderBy(F.col("shipment_cost").desc())

top3 = df.withColumn("rank", F.dense_rank().over(w)) \
         .filter("rank <= 3")

# LEAD / LAG
lag_df = df.withColumn(
    "prev_shipment_days",
    F.lag("shipment_year").over(
        Window.partitionBy("masked_staff_name").orderBy("shipment_year")
    )
)
     

In [0]:

# CUBE AGGREGATION
cube_df = df.cube("origin_hub_city") \
    .agg(F.sum("shipment_cost").alias("total_cost"))

write_file(top3,f"{GOLD}/top3_drivers")
write_file(lag_df,f"{GOLD}/prev_shipment_days")
write_file(cube_df,f"{GOLD}/cube_costs")